# Lecture 04 exercises — Imbalanced learning

Starter cells run. Answer prompts in Markdown; TODO cells are safe extension points.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    auc,
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    brier_score_loss,
)
from xgboost import XGBClassifier


def repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "data" / "fraud.csv").exists():
            return path
    raise FileNotFoundError("Could not find data/fraud.csv")


ROOT = repo_root()
data = pd.read_csv(ROOT / "data" / "fraud.csv")
features = [
    "amount",
    "hour",
    "card_present",
    "international",
    "account_age_days",
    "transactions_24h",
    "merchant_risk",
]
X, y = data[features], data["fraud"].astype(int)
print(f"rows={len(data)}, positives={y.sum()}, prevalence={y.mean():.3%}")
# Split first. The test rows remain untouched until all choices are frozen.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=14, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=15, stratify=y_dev
)
print("sizes:", len(X_train), len(X_valid), len(X_test))


def empirical_cost(actual, scores, threshold, c_fn=20.0, c_fp=1.0):
    predicted = scores >= threshold
    false_negative = np.sum((actual.to_numpy() == 1) & ~predicted)
    false_positive = np.sum((actual.to_numpy() == 0) & predicted)
    return c_fn * false_negative + c_fp * false_positive

## Exercise 1 — Majority baseline

Calculate test prevalence, accuracy and recall for an always-negative rule. Explain the misleading accuracy.

In [ ]:
negative, positive = np.bincount(y_test, minlength=2)
print(
    {
        "prevalence": positive / len(y_test),
        "accuracy": negative / len(y_test),
        "recall": 0.0,
    }
)
# TODO: add an interpretation in a Markdown cell.

## Exercise 2 — Posterior shift

Derive $q/(1-q)=(w_1/w_0)p/(1-p)$. Find the natural probability for which weighted $q=1/2$.

In [ ]:
p = np.linspace(0.001, 0.2, 100)
w0, w1 = 1.0, 10.0
q = w1 * p / (w1 * p + w0 * (1 - p))
plt.plot(p, q)
plt.axhline(0.5, color="black", linestyle="--")
plt.axvline(w0 / (w0 + w1), color="black", linestyle=":")
plt.xlabel("natural probability p")
plt.ylabel("weighted-risk q")
# TODO: explain why q=.5 differs from p=.5.

## Exercise 3 — Compare metrics

Fit weighted and unweighted logistic regression. Compare AP, trapezoidal PR area, ROC-AUC and Brier on validation.

In [ ]:
estimators = {
    "unweighted": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000, random_state=14)
    ),
    "balanced": make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight="balanced", max_iter=2000, random_state=14),
    ),
}
for estimator in estimators.values():
    estimator.fit(X_train, y_train)
for name, estimator in estimators.items():
    scores = estimator.predict_proba(X_valid)[:, 1]
    precision, recall, _ = precision_recall_curve(y_valid, scores)
    print(
        name,
        {
            "AP": average_precision_score(y_valid, scores),
            "trapezoid": auc(recall[::-1], precision[::-1]),
            "ROC_AUC": roc_auc_score(y_valid, scores),
            "Brier": brier_score_loss(y_valid, scores),
        },
    )

## Exercise 4 — Threshold policy

Choose a threshold for false-negative cost 20 and false-positive cost 1 using validation. Freeze it and inspect test once; explain why test cannot choose it.

In [ ]:
scores = estimators["unweighted"].predict_proba(X_valid)[:, 1]
grid = np.unique(np.r_[0.0, np.linspace(0.01, 0.99, 197), 1.0])
costs = [empirical_cost(y_valid, scores, t, 20, 1) for t in grid]
threshold = float(grid[int(np.argmin(costs))])
print("validation threshold:", threshold, "formula if calibrated:", 1 / 21)
# TODO: report the test confusion matrix using this frozen threshold.